%% [markdown]
# Bước 1: Thu thập dữ liệu Sách từ Tiki API

**Mục tiêu:** Thu thập tối thiểu 5.000 sản phẩm sách từ các danh mục khác nhau trên Tiki
thông qua Public API (không cần đăng nhập).
#
**Chiến lược 2 bước:**
- Bước A: Lấy danh sách `product_id` từ API danh mục
- Bước B: Lấy thông tin chi tiết từng sản phẩm qua API detail
#
**Thư viện sử dụng:** `requests`, `pandas`, `time`, `json`, `tqdm`

## 0. Import thư viện

In [1]:
import requests
import pandas as pd
import numpy as np
import time
import json
import os
import random
from datetime import datetime
from tqdm import tqdm  # pip install tqdm nếu chưa có

# Tắt cảnh báo SSL không cần thiết
import warnings
warnings.filterwarnings("ignore")

print("✅ Import thư viện thành công")
print(f"📅 Thời điểm bắt đầu crawl: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

✅ Import thư viện thành công
📅 Thời điểm bắt đầu crawl: 2026-04-15 20:10:47


## 1. Cấu hình chung

=====================================================================
CẤU HÌNH CRAWL
=====================================================================

Đường dẫn lưu file
RAW_DATA_PATH = "../data/raw/tiki_books_raw.csv"
CHECKPOINT_PATH = "../data/raw/product_ids_checkpoint.json"

Headers giả lập trình duyệt thật - giúp tránh bị block
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    ),
    "Accept": "application/json, text/plain, */*",
    "Accept-Language": "vi-VN,vi;q=0.9,en-US;q=0.8,en;q=0.7",
    "Accept-Encoding": "gzip, deflate, br",
    "Referer": "https://tiki.vn/",
    "Origin": "https://tiki.vn",
    "Connection": "keep-alive",
}

Delay giữa các request (giây) - để không bị block IP
DELAY_BETWEEN_REQUESTS = (0.5, 1.5)  # Random trong khoảng này
DELAY_BETWEEN_CATEGORIES = (2, 4)

Số trang tối đa mỗi danh mục khi lấy ID
MAX_PAGES_PER_CATEGORY = 50  # 40 sản phẩm/trang × 50 trang = 2.000 ID/danh mục

=====================================================================
DANH MỤC SÁCH TRÊN TIKI (category_id: tên danh mục)
=====================================================================
ID danh mục lấy từ URL khi duyệt tiki.vn
BOOK_CATEGORIES = {
    "Sách Văn Học":         7,
    "Sách Thiếu Nhi":       11,
    "Kỹ Năng Sống":         676768,
    "Sách Kinh Tế":         693611,
    "Sách Khoa Học":        275535,
    "Sách Giáo Khoa":       11290,
    "Manga - Truyện Tranh": 27703,
    "Sách Tâm Lý":          320,
}

print(f"📚 Số danh mục sẽ crawl: {len(BOOK_CATEGORIES)}")
for name, cat_id in BOOK_CATEGORIES.items():
    print(f"   - {name}: ID = {cat_id}")

In [ ]:
#=====================================================================
#CẤU HÌNH CRAWL
#=====================================================================

#Đường dẫn lưu file
RAW_DATA_PATH = "../data/raw/tiki_books_raw.csv"
CHECKPOINT_PATH = "../data/raw/product_ids_checkpoint.json"

#Headers giả lập trình duyệt thật - giúp tránh bị block
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    ),
    "Accept": "application/json, text/plain, */*",
    "Accept-Language": "vi-VN,vi;q=0.9,en-US;q=0.8,en;q=0.7",
    "Accept-Encoding": "gzip, deflate, br",
    "Referer": "https://tiki.vn/",
    "Origin": "https://tiki.vn",
    "Connection": "keep-alive",
}

#Delay giữa các request (giây) - để không bị block IP
DELAY_BETWEEN_REQUESTS = (0.5, 1.5)  # Random trong khoảng này
DELAY_BETWEEN_CATEGORIES = (2, 4)

#Số trang tối đa mỗi danh mục khi lấy ID
MAX_PAGES_PER_CATEGORY = 50  # 40 sản phẩm/trang × 50 trang = 2.000 ID/danh mục

#=====================================================================
#DANH MỤC SÁCH TRÊN TIKI (category_id: tên danh mục)
#=====================================================================
ID danh mục lấy từ URL khi duyệt tiki.vn
BOOK_CATEGORIES = {
    "Sách Văn Học":         7,
    "Sách Thiếu Nhi":       11,
    "Kỹ Năng Sống":         676768,
    "Sách Kinh Tế":         693611,
    "Sách Khoa Học":        275535,
    "Sách Giáo Khoa":       11290,
    "Manga - Truyện Tranh": 27703,
    "Sách Tâm Lý":          320,
}

print(f"📚 Số danh mục sẽ crawl: {len(BOOK_CATEGORIES)}")
for name, cat_id in BOOK_CATEGORIES.items():
    print(f"   - {name}: ID = {cat_id}")

## 2. Các hàm tiện ích

In [2]:
def safe_get(url: str, params: dict = None, retries: int = 3) -> dict | None:
    """
    Gọi GET request an toàn với cơ chế retry.
    
    Args:
        url: URL cần gọi
        params: Dict các tham số query string
        retries: Số lần thử lại nếu thất bại
    
    Returns:
        Dict JSON hoặc None nếu thất bại
    """
    for attempt in range(retries):
        try:
            resp = requests.get(
                url,
                headers=HEADERS,
                params=params,
                timeout=15,
                verify=True
            )
            if resp.status_code == 200:
                return resp.json()
            elif resp.status_code == 429:
                # Rate limit - chờ lâu hơn
                wait_time = 10 + attempt * 5
                print(f"   ⚠️  Rate limit (429). Chờ {wait_time}s...")
                time.sleep(wait_time)
            elif resp.status_code in [403, 404]:
                return None  # Không retry nếu bị từ chối hoặc không tồn tại
            else:
                print(f"   ⚠️  HTTP {resp.status_code} - Thử lại ({attempt+1}/{retries})")
                time.sleep(2 ** attempt)
        except requests.exceptions.Timeout:
            print(f"   ⏳ Timeout - Thử lại ({attempt+1}/{retries})")
            time.sleep(2)
        except requests.exceptions.ConnectionError:
            print(f"   🔌 Connection error - Thử lại ({attempt+1}/{retries})")
            time.sleep(3)
        except Exception as e:
            print(f"   ❌ Lỗi không mong đợi: {e}")
            return None
    return None


# ✅ Đúng: None làm default, lấy giá trị thật khi hàm được GỌI
def random_delay(delay_range: tuple = None):
    if delay_range is None:
        delay_range = DELAY_BETWEEN_REQUESTS   # lấy từ biến toàn cục lúc gọi
    delay = random.uniform(*delay_range)
    time.sleep(delay)



print("✅ Định nghĩa hàm tiện ích thành công")

✅ Định nghĩa hàm tiện ích thành công


## 3. Bước A — Thu thập danh sách Product ID theo danh mục

In [3]:
def get_product_ids_from_category(category_id: int, category_name: str, max_pages: int = MAX_PAGES_PER_CATEGORY) -> list[int]:
    """
    Lấy tất cả product_id từ một danh mục sách trên Tiki.
    
    Args:
        category_id: ID danh mục trên Tiki
        category_name: Tên hiển thị để log
        max_pages: Số trang tối đa mỗi danh mục
    
    Returns:
        List các product_id
    """
    BASE_URL = "https://tiki.vn/api/personalish/v1/blocks/listings"
    
    all_ids = []
    
    print(f"\n📂 Đang lấy ID từ danh mục: [{category_name}] (ID={category_id})")
    
    for page in range(1, max_pages + 1):
        params = {
            "limit":       40,
            "include":     "advertisement",
            "aggregations": 2,
            "trackity_id": "dummy-tracking-id",
            "category":    category_id,
            "page":        page,
            "urlKey":      "nha-sach-tiki",
        }
        
        data = safe_get(BASE_URL, params=params)
        
        if data is None:
            print(f"   ⚠️  Không lấy được trang {page}. Dừng danh mục này.")
            break
        
        # Lấy danh sách sản phẩm trong trang
        items = data.get("data", [])
        
        if not items:
            print(f"   ✅ Đã hết sản phẩm tại trang {page}. Tổng: {len(all_ids)} ID")
            break
        
        # Trích xuất product_id
        page_ids = [item["id"] for item in items if "id" in item]
        all_ids.extend(page_ids)
        
        print(f"   📄 Trang {page:02d}: +{len(page_ids)} IDs → Tổng: {len(all_ids)}")
        
        random_delay(DELAY_BETWEEN_REQUESTS)
    
    return list(set(all_ids))  # Loại bỏ duplicate ngay từ đầu

NameError: name 'MAX_PAGES_PER_CATEGORY' is not defined

Chạy thu thập ID từ tất cả danh mục
Nếu đã có checkpoint, bỏ qua bước này

all_product_ids = {}  # {category_name: [product_ids]}

if os.path.exists(CHECKPOINT_PATH):
    print(f"📂 Tìm thấy checkpoint tại {CHECKPOINT_PATH}")
    print("   → Load lại danh sách ID đã thu thập trước đó...")
    with open(CHECKPOINT_PATH, "r", encoding="utf-8") as f:
        all_product_ids = json.load(f)
    
    total_loaded = sum(len(v) for v in all_product_ids.values())
    print(f"✅ Đã load {total_loaded} product IDs từ {len(all_product_ids)} danh mục")
else:
    print("🚀 Bắt đầu thu thập product IDs từ các danh mục...")
    
    for cat_name, cat_id in BOOK_CATEGORIES.items():
        ids = get_product_ids_from_category(cat_id, cat_name)
        all_product_ids[cat_name] = ids
        
        # Lưu checkpoint sau mỗi danh mục
        os.makedirs(os.path.dirname(CHECKPOINT_PATH), exist_ok=True)
        with open(CHECKPOINT_PATH, "w", encoding="utf-8") as f:
            json.dump(all_product_ids, f, ensure_ascii=False)
        
        print(f"   💾 Đã lưu checkpoint.")
        random_delay(DELAY_BETWEEN_CATEGORIES)
    
    print("\n" + "="*60)
    print("📊 TỔNG KẾT THU THẬP ID:")
    total_ids = 0
    for cat_name, ids in all_product_ids.items():
        print(f"   {cat_name}: {len(ids)} IDs")
        total_ids += len(ids)
    print(f"   ──────────────────────")
    print(f"   TỔNG CỘNG: {total_ids} product IDs")

## 4. Bước B — Thu thập chi tiết từng sản phẩm

In [ ]:
def extract_product_detail(data: dict, category_name: str) -> dict | None:
    """
    Trích xuất các trường cần thiết từ JSON chi tiết sản phẩm Tiki.
    
    Args:
        data: Dict JSON từ Tiki API
        category_name: Tên danh mục để gán nhãn
    
    Returns:
        Dict với các trường đã chuẩn hóa
    """
    if not data or "id" not in data:
        return None
    
    try:
        # --- Thông tin cơ bản ---
        product_id   = data.get("id")
        name         = data.get("name", "").strip()
        
        # --- Giá ---
        price        = data.get("price", 0)
        list_price   = data.get("list_price", price)
        discount_rate = data.get("discount_rate", 0)
        
        # Tính lại discount nếu API không trả về
        if discount_rate == 0 and list_price > 0 and price < list_price:
            discount_rate = round((1 - price / list_price) * 100, 1)
        
        # --- Đánh giá ---
        rating_average = data.get("rating_average", 0)
        review_count   = data.get("review_count", 0)
        
        # --- Doanh số ---
        qty_sold_obj = data.get("quantity_sold", {})
        if isinstance(qty_sold_obj, dict):
            quantity_sold = qty_sold_obj.get("value", 0)
        elif isinstance(qty_sold_obj, (int, float)):
            quantity_sold = int(qty_sold_obj)
        else:
            quantity_sold = 0
        
        # --- Thông tin sách ---
        # Lấy author_name từ nhiều vị trí có thể có
        author_name = data.get("author_name", "")
        if not author_name:
            # Thử lấy từ specifications
            specs = data.get("specifications", [])
            for spec in specs:
                for attr in spec.get("attributes", []):
                    if attr.get("code") in ["author", "tac_gia"]:
                        author_name = attr.get("value", "")
                        break
        
        publisher_name = data.get("publisher_name", "")
        if not publisher_name:
            specs = data.get("specifications", [])
            for spec in specs:
                for attr in spec.get("attributes", []):
                    if attr.get("code") in ["publisher", "nha_xuat_ban"]:
                        publisher_name = attr.get("value", "")
                        break
        
        # --- Thể loại ---
        # Ưu tiên category thực tế từ API, fallback sang tên danh mục đã biết
        breadcrumbs = data.get("breadcrumbs", [])
        if breadcrumbs and len(breadcrumbs) >= 2:
            actual_category = breadcrumbs[-2].get("name", category_name)
        else:
            actual_category = category_name
        
        # --- Tình trạng kho ---
        inventory_status = data.get("inventory_status", "unknown")
        
        # --- URL sản phẩm ---
        url_path = data.get("url_path", "")
        product_url = f"https://tiki.vn/{url_path}" if url_path else ""
        
        # --- Hình ảnh đại diện ---
        thumbnail = data.get("thumbnail_url", "")
        
        return {
            "product_id":       product_id,
            "name":             name,
            "price":            price,
            "list_price":       list_price,
            "discount_rate":    discount_rate,
            "rating_average":   rating_average,
            "review_count":     review_count,
            "quantity_sold":    quantity_sold,
            "author_name":      author_name,
            "publisher_name":   publisher_name,
            "category_name":    actual_category,
            "crawl_category":   category_name,
            "inventory_status": inventory_status,
            "product_url":      product_url,
            "thumbnail_url":    thumbnail,
            "crawl_time":       datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        }
    
    except Exception as e:
        print(f"   ⚠️  Lỗi parse sản phẩm {data.get('id', '?')}: {e}")
        return None


def crawl_product_detail(product_id: int, category_name: str) -> dict | None:
    """Gọi API Tiki để lấy chi tiết một sản phẩm."""
    url = f"https://tiki.vn/api/v2/products/{product_id}"
    
    data = safe_get(url)
    if data is None:
        return None
    
    return extract_product_detail(data, category_name)


print("✅ Định nghĩa hàm crawl chi tiết thành công")

Tạo danh sách tất cả (product_id, category_name) để crawl chi tiết
all_pairs = []
for cat_name, ids in all_product_ids.items():
    for pid in ids:
        all_pairs.append((pid, cat_name))

Xáo trộn để không crawl tuần tự cùng danh mục (giảm nguy cơ bị block)
random.shuffle(all_pairs)

print(f"📋 Tổng số sản phẩm cần crawl chi tiết: {len(all_pairs)}")

Kiểm tra xem đã có dữ liệu chưa (tiếp tục nếu bị gián đoạn)
existing_records = []
crawled_ids = set()

if os.path.exists(RAW_DATA_PATH):
    df_existing = pd.read_csv(RAW_DATA_PATH, encoding="utf-8-sig")
    existing_records = df_existing.to_dict("records")
    crawled_ids = set(df_existing["product_id"].astype(int).tolist())
    print(f"📂 Tìm thấy {len(crawled_ids)} sản phẩm đã crawl trước đó → Bỏ qua và tiếp tục")
else:
    print("🆕 Bắt đầu crawl mới từ đầu")

Lọc ra những ID chưa crawl
remaining_pairs = [(pid, cat) for pid, cat in all_pairs if pid not in crawled_ids]
print(f"🔄 Số sản phẩm còn lại cần crawl: {len(remaining_pairs)}")

## 5. Vòng lặp crawl chính

=====================================================================
VÒNG LẶP CRAWL CHI TIẾT SẢN PHẨM
=====================================================================
SAVE_EVERY = 100  # Lưu file mỗi N sản phẩm

successful = 0
failed = 0
records = list(existing_records)  # Bắt đầu từ dữ liệu đã có

print(f"\n🚀 Bắt đầu crawl chi tiết {len(remaining_pairs)} sản phẩm...")
print(f"   Lưu checkpoint mỗi {SAVE_EVERY} sản phẩm")
print("="*60)

for i, (product_id, category_name) in enumerate(tqdm(remaining_pairs, desc="Crawling")):
    
    detail = crawl_product_detail(product_id, category_name)
    
    if detail is not None:
        records.append(detail)
        successful += 1
    else:
        failed += 1
    
    # Lưu file mỗi SAVE_EVERY sản phẩm
    if (i + 1) % SAVE_EVERY == 0:
        df_temp = pd.DataFrame(records)
        os.makedirs(os.path.dirname(RAW_DATA_PATH), exist_ok=True)
        df_temp.to_csv(RAW_DATA_PATH, index=False, encoding="utf-8-sig")
        tqdm.write(f"   💾 [{i+1}/{len(remaining_pairs)}] Đã lưu {len(records)} bản ghi | Thành công: {successful} | Thất bại: {failed}")
    
    # Delay ngẫu nhiên
    random_delay(DELAY_BETWEEN_REQUESTS)

Lưu lần cuối
df_raw = pd.DataFrame(records)
os.makedirs(os.path.dirname(RAW_DATA_PATH), exist_ok=True)
df_raw.to_csv(RAW_DATA_PATH, index=False, encoding="utf-8-sig")

print("\n" + "="*60)
print("🎉 CRAWL HOÀN TẤT!")
print(f"   ✅ Thành công:  {successful} sản phẩm")
print(f"   ❌ Thất bại:    {failed} sản phẩm")
print(f"   📊 Tổng bản ghi: {len(df_raw)}")
print(f"   💾 Đã lưu tại:  {RAW_DATA_PATH}")

## 6. Kiểm tra sơ bộ dữ liệu thô

Load lại file vừa lưu để kiểm tra
df_raw = pd.read_csv(RAW_DATA_PATH, encoding="utf-8-sig")

print("=" * 60)
print("📊 TỔNG QUAN DỮ LIỆU THÔ:")
print("=" * 60)
print(f"  Số hàng:    {df_raw.shape[0]:,}")
print(f"  Số cột:     {df_raw.shape[1]}")
print(f"  Các cột:    {list(df_raw.columns)}")

In [ ]:
print("\n📋 5 dòng đầu:")
df_raw.head()

In [ ]:
print("\n📋 Thông tin kiểu dữ liệu:")
df_raw.info()

In [ ]:
print("\n📋 Thống kê mô tả các cột số:")
df_raw.describe()

In [ ]:
print("\n📋 Số lượng sản phẩm theo danh mục:")
df_raw["crawl_category"].value_counts()

In [ ]:
print("\n📋 Tỷ lệ giá trị thiếu:")
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)
pd.DataFrame({"Thiếu": missing, "Tỷ lệ (%)": missing_pct})[missing > 0]

Kiểm tra đủ 5000 dòng chưa
target_rows = 5000
current_rows = len(df_raw)

if current_rows >= target_rows:
    print(f"\n✅ ĐẠT YÊU CẦU: {current_rows:,} dòng ≥ {target_rows:,} dòng yêu cầu")
else:
    deficit = target_rows - current_rows
    print(f"\n⚠️  CHƯA ĐỦ: {current_rows:,} dòng. Cần thêm {deficit:,} dòng nữa.")
    print("   → Hãy thêm danh mục hoặc tăng MAX_PAGES_PER_CATEGORY và chạy lại")

In [ ]:
print("\n✅ Bước 1 (Thu thập dữ liệu) hoàn thành!")
print(f"📁 File thô lưu tại: {RAW_DATA_PATH}")
print("→ Chuyển sang Notebook 02_preprocessing.ipynb để làm sạch dữ liệu")